In [1]:
import torch
from datamodules.bray_data_module import BrayDataModule
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Module, ModuleList
from loss.masked_bce_loss import MaskedBCELoss
from t

In [2]:
bray_datamodule = BrayDataModule(
    batch_size=16, data_dir="../_data/gigadb", mask_uncertain=True
)

Bray DM init
../_data/gigadb/gigadb.csv


In [5]:
bray_datamodule.setup(stage="fit")

Bray DM setup
Loading dataset using Polars lazy frames...
Identifying MoA columns...
Identifying feature columns...
Building lazy query for efficient data loading...
Executing query and loading data...
Loaded 35639 samples
Converting to numpy arrays...
Target Distribution Summary:
Total target values: 178195
Total 1s (positive): 9981 (5.60%)
Total 0s (negative): 71340 (40.03%)
Total uncertain: 96874 (54.36%)
Positive-to-Negative ratio: 0.1399
Dataset loaded with 35639 samples
Feature dimension: 4643
Target dimension: 5
Loading dataset using Polars lazy frames...
Identifying MoA columns...
Identifying feature columns...
Building lazy query for efficient data loading...
Executing query and loading data...
Loaded 11371 samples
Converting to numpy arrays...
Target Distribution Summary:
Total target values: 56855
Total 1s (positive): 3101 (5.45%)
Total 0s (negative): 22577 (39.71%)
Total uncertain: 31177 (54.84%)
Positive-to-Negative ratio: 0.1374
Dataset loaded with 11371 samples
Feature d

In [6]:
class DiffusionAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding_dim = 668
        self.num_classes = 5
        self.input_dim = self.embedding_dim + self.num_classes + 1

        #embedding layer
        self.embedding_layer = nn.Linear(self.input_dim, 512)

        #model
        self.activation = nn.ReLU()
        self.sigmoid = nn.Sigmoid()
        
        self.fc1 = nn.Linear(512, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, 32)
        self.fc5 = nn.Linear(32, 16)

        self.fc6 = nn.Linear(16, 32)
        self.fc7 = nn.Linear(32, 64)
        self.fc8 = nn.Linear(64, 128)
        self.fc9 = nn.Linear(128, 256)
        self.output = nn.Linear(256, self.num_classes)


    def forward(self, x, y, ts):
        x = torch.cat([x, y, ts], dim=1)
        x = self.embedding_layer(x)
        x = self.fc1(x)
        x = self.activation(x)
        x = self.fc2(x)
        x = self.activation(x)
        x = self.fc3(x)
        x = self.activation(x)
        x = self.fc4(x)
        x = self.activation(x)
        x = self.fc5(x)
        x = self.activation(x)
        x = self.fc6(x)
        x = self.activation(x)
        x = self.fc7(x)
        x = self.activation(x)
        x = self.fc8(x)
        x = self.activation(x)
        x = self.fc9(x)
        x = self.activation(x)
        x = self.output(x)
        x = self.sigmoid(x)

        return x
        

In [11]:
dataloader = bray_datamodule.train_dataloader()

In [10]:
autoencoder = DiffusionAutoencoder()

In [15]:
for idx, batch in enumerate(dataloader):
    x, y, mask = batch
    y_pred = autoencoder(x, y, 0)

RuntimeError: DataLoader worker (pid(s) 4039, 4040, 4041, 4042) exited unexpectedly